In [2]:
import pandas as pd
import re
import os

print("🚀 Starting FINAL Master ETL Pipeline...")

# ==========================================
# PHASE 1: EXTRACT
# ==========================================
file_path = 'A_Z_medicines_dataset_of_India.csv'

if not os.path.exists(file_path):
    print(f"❌ File not found: {file_path}")
    exit()

df_raw = pd.read_csv(file_path)

# ==========================================
# PHASE 2: TRANSFORM
# ==========================================

def classify_medicine_final(row):

    name = str(row['name']).lower()
    pack = str(row['pack_size_label']).lower()
    text = f"{name} {pack}"

    tokens = re.split(r'[\s\-/()]+', text)

    def has_token(words):
        return any(w in tokens for w in words)

    def has_ds():
        return bool(re.search(r'(\bds\b|\bdds\b|-ds\b|-dds\b)', text))

    def last_token_capsule():
        last = tokens[-1] if tokens else ""
        return any(x in last for x in [
            "cap", "caps", "instacap", "redicap",
            "transcap", "transcaps", "octacap",
            "easecaps", "divicap", "respicap"
        ])

    # -------------------------
    # SPECIALIZED
    # -------------------------
    if has_token(['inhaler', 'respules', 'rotacap']):
        return 'Specialized: Inhaler'

    if has_token(['spray']):
        return 'Specialized: Spray'

    if has_token(['gargle']):
        return 'Specialized: Gargle'

    if has_token(['wash', 'scrub']):
        return 'Specialized: Wash'

    if has_token(['powder', 'kit']):
        return 'Specialized: Others'

    # -------------------------
    # INJECTION
    # -------------------------
    if has_token(['injection', 'infusion', 'vial', 'ampoule']):
        return 'Internal: Injection'

    # -------------------------
    # CAPSULE (PRIORITY)
    # -------------------------
    if has_token(['capsule']) or last_token_capsule():

        if has_token(['sr', 'er', 'cr', 'mr', 'xr']):
            return 'Internal: Capsule (Modified Release)'

        if has_token(['ec', 'enteric']):
            return 'Internal: Capsule (Enteric Coated)'

        if has_token(['softgel', 'sgc']):
            return 'Internal: Capsule (Soft Gelatin)'

        if has_token(['hpmc', 'veg']):
            return 'Internal: Capsule (Vegetarian HPMC)'

        return 'Internal: Capsule (Hard Gelatin)'

    # -------------------------
    # TABLET
    # -------------------------
    if has_token(['tablet', 'tab', 'caplet']):

        if has_token(['sr']):
            return 'Internal: Tablet (SR)'

        if has_token(['er', 'xr']):
            return 'Internal: Tablet (ER)'

        if has_token(['dt', 'dispersible']):
            return 'Internal: Tablet (DT)'

        if has_token(['md', 'od', 'dissolving']):
            return 'Internal: Tablet (MD)'

        return 'Internal: Tablet (Plain)'

    # -------------------------
    # DRY SYRUP
    # -------------------------
    if has_ds():
        return 'Internal: Dry Syrup'

    # -------------------------
    # SUSPENSION
    # -------------------------
    if has_token(['suspension']):
        if has_token(['eye', 'ear', 'otic', 'ophthalmic']):
            return 'External: Ophthalmic Otic Suspension'
        return 'Internal: Oral Suspension'

    # -------------------------
    # EXTERNAL
    # -------------------------
    if has_token(['cream']):
        return 'External: Cream'

    if has_token(['ointment']):
        return 'External: Ointment'

    if has_token(['gel']) and not has_token(['softgel']):
        return 'External: Gel'

    if has_token(['lotion']):
        return 'External: Lotion'

    if has_token(['patch']):
        return 'External: Patch'

    # -------------------------
    # DROPS
    # -------------------------
    if has_token(['drop', 'drops']):
        if has_token(['eye', 'ear', 'otic', 'ophthalmic']):
            return 'External: Eye Ear Drops'
        if has_token(['oral']):
            return 'Internal: Oral Drops'
        return 'Unknown'

    # -------------------------
    # LIQUID
    # -------------------------
    if has_token(['syrup', 'liquid']):
        return 'Internal: Oral Liquid'

    return 'Unknown'


# ==========================================
# SALT EXTRACTION
# ==========================================

def parse_composition(comp_str):
    if pd.isna(comp_str):
        return []

    parts = str(comp_str).split(' , ')
    results = []

    for p in parts:
        match = re.search(r'^(.*?)\s*\((.*?)\)$', p.strip())
        if match:
            results.append((match.group(1).strip(), match.group(2).strip()))
        else:
            results.append((p.strip(), "NA"))

    return results


# ==========================================
# PRICE PER UNIT
# ==========================================

def extract_qty(label):
    match = re.search(r'of\s+(\d+\.?\d*)', str(label).lower())
    if match:
        return float(match.group(1))

    match = re.search(r'^(\d+)', str(label).lower())
    if match:
        return float(match.group(1))

    return 1.0


# ==========================================
# APPLY TRANSFORM
# ==========================================

print("🧠 Applying FINAL Classification Engine...")
df_raw['Header_Category'] = df_raw.apply(classify_medicine_final, axis=1)

print("🔬 Extracting Salt Data...")
salts = df_raw.apply(
    lambda row: (parse_composition(row['short_composition1']) +
                 parse_composition(row['short_composition2']) +
                 [("NA", "NA")] * 3)[:3],
    axis=1
)

df_raw['Salt 1 Name'] = [s[0][0] for s in salts]
df_raw['Salt 1 Strength'] = [s[0][1] for s in salts]
df_raw['Salt 2 Name'] = [s[1][0] for s in salts]
df_raw['Salt 2 Strength'] = [s[1][1] for s in salts]
df_raw['Salt 3 Name'] = [s[2][0] for s in salts]
df_raw['Salt 3 Strength'] = [s[2][1] for s in salts]

print("💰 Calculating price per unit...")
df_raw['qty'] = df_raw['pack_size_label'].apply(extract_qty)
df_raw['price_per_unit'] = df_raw['price(₹)'] / df_raw['qty']
df_raw['Source'] = 'Market'

final_columns = [
    'name', 'manufacturer_name',
    'Salt 1 Name', 'Salt 1 Strength',
    'Salt 2 Name', 'Salt 2 Strength',
    'Salt 3 Name', 'Salt 3 Strength',
    'Header_Category', 'price(₹)', 'price_per_unit', 'Source'
]

df_clean = df_raw[final_columns]

# ==========================================
# EXPORT
# ==========================================

print("💾 Exporting files...")

trash_df = df_clean[df_clean['Header_Category'] == 'Unknown']
trash_df.to_csv('Trash_Medicines_Audit.csv', index=False)

df_valid = df_clean[df_clean['Header_Category'] != 'Unknown']

def clean_sheet_name(name, existing_names):
    name = name.split(':')[-1].strip()

    forbidden = [':', '/', '\\', '?', '*', '[', ']']
    for char in forbidden:
        name = name.replace(char, ' ')

    name = re.sub(r'\s+', ' ', name).strip()
    name = name[:31]

    original = name
    i = 1
    while name in existing_names:
        name = f"{original[:28]}_{i}"
        i += 1

    existing_names.add(name)
    return name


def save_excel(prefix, filename):
    data = df_valid[df_valid['Header_Category'].str.startswith(prefix, na=False)]

    existing_names = set()

    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        for cat in data['Header_Category'].unique():
            sheet = clean_sheet_name(cat, existing_names)
            data[data['Header_Category'] == cat].to_excel(writer, sheet_name=sheet, index=False)

    print(f"✅ Saved {filename}")


save_excel('Internal:', 'Internal_Medicines_FINAL.xlsx')
save_excel('External:', 'External_Medicines_FINAL.xlsx')
save_excel('Specialized:', 'Specialized_Medicines_FINAL.xlsx')

print("🎉 FINAL PIPELINE COMPLETE")

🚀 Starting FINAL Master ETL Pipeline...
🧠 Applying FINAL Classification Engine...
🔬 Extracting Salt Data...
💰 Calculating price per unit...
💾 Exporting files...
✅ Saved Internal_Medicines_FINAL.xlsx
✅ Saved External_Medicines_FINAL.xlsx
✅ Saved Specialized_Medicines_FINAL.xlsx
🎉 FINAL PIPELINE COMPLETE
